In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import os

In [4]:
PROJECT_PATH = '/content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill'

os.listdir(PROJECT_PATH)
!wc -l {PROJECT_PATH}/data/train.jsonl {PROJECT_PATH}/data/test.jsonl

  120 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/train.jsonl
   30 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/test.jsonl
  150 total


In [5]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": f"{PROJECT_PATH}/data/train.jsonl",
    "test": f"{PROJECT_PATH}/data/test.jsonl"
})

print(dataset)
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 120
    })
    test: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 30
    })
})
{'input': 'Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.', 'output': {'symptoms': ['chest pain', 'shortness of breath'], 'duration': ['2 hours'], 'severity': ['severe'], 'urgent': True}, 'metadata': {'clinical_domain': 'cardiac', 'generated_by': 'gpt-4o'}}


Format for Instruction Tuning


WHY WE FORMAT THE DATASET INTO A "text" FIELD
Our raw dataset has 3 separate fields:
  - "input"    → the clinical note text
  - "output"   → the correct structured JSON answer
  - "metadata" → domain, source model info
#
SFTTrainer (the fine-tuning library) needs ONE single string
 per example to feed into the model during training.

 So we combine input + output into a single "text" field
 using the Alpaca instruction format:

 ### Instruction:  → tells model what task to do
  ### Input:        → the clinical note
   ### Output:       → the correct JSON extraction

The model reads thousands of these and learns the pattern:
 "when I see a clinical note → produce structured JSON"
 This is standard instruction-tuning format used in:
Alpaca, LLaMA fine-tuning, and most clinical NLP papers.

 dataset_text_field="text" in SFTTrainer tells the trainer
which field to use for training.


In [6]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
def format_example(example):
  return {
      "text": f""" Instruction:

      Extract structured clinical information from the following text.

      ### Input:
      {example['input']}

      ### Output:
      {str(example['output'])}"""
  }

dataset = dataset.map(format_example)
print(dataset["train"][0]["text"])

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

 Instruction:

      Extract structured clinical information from the following text.

      ### Input:
      Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.

      ### Output:
      {'symptoms': ['chest pain', 'shortness of breath'], 'duration': ['2 hours'], 'severity': ['severe'], 'urgent': True}


In [9]:
# ============================================================
# LOAD MODEL + TOKENIZER
# ============================================================
# We use Gemma-3-1B — smallest Gemma that fits on free T4 GPU
# float16 reduces memory usage vs float32
# device_map="auto" automatically puts model on GPU
# ============================================================


from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype = torch.float16,
    device_map="auto"
)

print(f"Model loaded: {model_id}")
print(f"Parameters: {model.num_parameters():,}")



config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model loaded: google/gemma-3-1b-it
Parameters: 999,885,952


Lora Configuration:

Instead of training all 1b parameters, LoRA injects tiny trainable adapters into the attention layers only

In [10]:
# r=8           → adapter rank (size of adapter, higher = more params)
# lora_alpha=16 → scaling factor, usually 2x r
# target_modules → q_proj & v_proj are the attention layers
#                  these are the most important for learning new tasks
# lora_dropout  → prevents overfitting on small dataset
# bias="none"   → don't train bias terms
# task_type     → we're doing causal language modeling
#
# Result: only ~0.15% of parameters train instead of 100%
# This is why LoRA fits on a free T4 GPU!

PEFT: parameter efficient fine-tuning: hugging face lib that contains mul techniques for fine-tuning models without training all parameters


Attention mech working:
Every time a model reads a word it asks three question:

Q (query) -> "What am i looking for?"
K (key) -> "what do i Contain?"
V (Value) -> What info do i pass forward?"

"Patient has chest pain and shortness of breath"

Q → "chest pain" asks → what else is related to me?
K → "shortness of breath" answers → I'm related, I'm a symptom too
V → passes → both should go into symptoms array

q_proj = the layer that creates Query vectors
v_proj = the layer that creates Value vectors

In [12]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



trainable params: 745,472 || all params: 1,000,631,424 || trainable%: 0.0745


# Training Arguments

### num_train_epochs=3     → 3 full passes through your 120 examples
### batch_size=2           → 2 examples at a time (T4 memory limit)
### gradient_accumulation  → simulates batch of 8 (2x4) without OOM
### learning_rate=2e-4     → standard for LoRA fine-tuning
### fp16=True              → matches our float16 model
### output_dir             → saves checkpoints to your Google Drive
### report_to="none"       → disables wandb logging

In [13]:
import torch
print(torch.cuda.is_available())       # should print True
print(torch.cuda.get_device_name(0))   # should print Tesla T4

True
Tesla T4


In [15]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl gradio huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.6 MB/s eta 0:00:00


In [22]:
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = f"{PROJECT_PATH}/models/gemma-lora"
args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
    # max_seq_length=512
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    # tokenizer=tokenizer
    processing_class=tokenizer,

)

print("Starting training...")
trainer.train()
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


Starting training...


Step,Training Loss
10,2.571583
20,1.867606
30,1.352617
40,0.979323
50,0.797288
60,0.755299
70,0.733636


Training complete!


In [21]:
import trl
print(trl.__version__)

1.1.0


In [23]:
SAVE_PATH = f"{PROJECT_PATH}/models/gemma-lora-final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Model saved to: {SAVE_PATH}")

Model saved to: /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/models/gemma-lora-final
